# 07 — DataFrames, Tables & Pandas Integration

> **📓 Notebook · Module 04 · Intermediate**
> *Master `st.dataframe`, `column_config`, filtering, and transforming data before display.*

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. Use `st.dataframe` with `column_config` to format numbers, currency, dates, and percentages.
2. Apply `hide_index`, `column_order`, and `height` for clean presentation.
3. Enable row selection with `on_select` for interactive data exploration.
4. Use Pandas `Styler` for conditional formatting within `st.dataframe`.
5. Build sidebar filters to transform DataFrames before display.
6. Aggregate, sort, pivot, and compute derived columns for meaningful presentation.

## 📋 Prerequisites

- Completed [Notebook 05 — Layouts & Containers](05_layouts_and_containers.ipynb)
- Pandas basics: DataFrame creation, indexing, groupby, merge
- NumPy basics: random data generation
- Understanding of Streamlit's rerun model and sidebar widgets

---

## 📚 Concept: The Display Spectrum

Streamlit provides multiple ways to show tabular data:

```
Static                                              Interactive
─────────────────────────────────────────────────────────────────
st.table()    st.write()    st.dataframe()    st.data_editor()
   │              │               │                   │
   ▼              ▼               ▼                   ▼
 HTML table   Auto-detect   Sortable, scrollable   Editable cells
 No scroll    No control    Column resize           Add/delete rows
 Fixed        Simple        Row selection           Download CSV
```

**Rule of thumb:** Start with `st.dataframe()`. Use `st.table()` only for tiny static tables.

## 🧠 Intuition: Tables Are the Bridge Between Data and Understanding

Raw data is a sea of numbers. A well-formatted table is a **map** that guides the user to the right answer.

Think of formatting as **signposts**:
- Currency format (`$1,234`) → "This is money"
- Percentage format (`45.2%`) → "This is a rate"
- Color coding (green/red) → "This is good/bad"
- Column order → "Read left to right, in this sequence"

The table itself doesn't change — but **how you present it** changes what the user understands.

---

## 🔧 Build It: Basic DataFrame Display

Start with the simplest case: creating and displaying a DataFrame.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="Notebook 07", layout="wide")

# Create a realistic dataset
np.random.seed(42)
n = 200

products = ["Laptop", "Phone", "Tablet", "Monitor", "Keyboard"]
regions = ["North", "South", "East", "West"]
categories = ["Electronics", "Accessories", "Peripherals"]

sales_df = pd.DataFrame({
    "Date": pd.date_range("2026-01-01", periods=n, freq="D"),
    "Product": np.random.choice(products, n),
    "Region": np.random.choice(regions, n),
    "Category": np.random.choice(categories, n),
    "Revenue": np.random.randint(500, 15000, n).astype(float),
    "Units": np.random.randint(1, 50, n),
    "Rating": np.random.uniform(1.0, 5.0, n).round(1),
})
sales_df["Unit Price"] = (sales_df["Revenue"] / sales_df["Units"]).round(2)

st.header("Raw Data")
st.dataframe(sales_df.head(10))

---

## 🔧 Build It: Clean Display with hide_index

In [ ]:
st.header("Clean Display with hide_index")

# Hide the index for a cleaner look
st.dataframe(sales_df, hide_index=True, height=350)

st.caption("The index column is hidden — the table looks cleaner and more focused.")

---

## 🔧 Build It: Column Configuration

`column_config` is how you make data **meaningful** to the reader.

In [ ]:
st.header("Column Configuration — Making Data Meaningful")

st.dataframe(
    sales_df,
    hide_index=True,
    column_config={
        "Date": st.column_config.DateColumn(
            "Sale Date",
            format="MMM D, YYYY",
        ),
        "Revenue": st.column_config.NumberColumn(
            "Revenue",
            format="$ %d",
            help="Total revenue for this transaction",
        ),
        "Unit Price": st.column_config.NumberColumn(
            "Unit Price",
            format="$ %.2f",
        ),
        "Rating": st.column_config.NumberColumn(
            "Rating",
            format="%.1f ⭐",
            min_value=1.0,
            max_value=5.0,
        ),
        "Units": st.column_config.NumberColumn(
            "Units Sold",
            format="%d",
        ),
    },
    column_order=["Date", "Product", "Region", "Category", "Revenue", "Units", "Unit Price", "Rating"],
    height=400,
)

st.caption("Numbers are formatted: revenue as currency, rating with stars, dates as 'Jan 1, 2026'.")

---

## 🔧 Build It: Hiding Columns

Not every column needs to be visible. Hide internal fields.

In [ ]:
st.header("Hiding Columns for Clean Presentation")

# Add an internal ID column
sales_df["_id"] = range(1, len(sales_df) + 1)

st.dataframe(
    sales_df,
    hide_index=True,
    column_config={
        "_id": None,  # Hide this column
        "Revenue": st.column_config.NumberColumn("Revenue", format="$ %d"),
    },
    column_order=["Date", "Product", "Region", "Revenue", "Units", "Rating"],
    height=350,
)

---

## 🔧 Build It: Row Selection

Make DataFrames interactive — let users select rows for details.

In [ ]:
st.header("Interactive Row Selection")
st.markdown("Click on rows to select them. The selected data appears below.")

selection = st.dataframe(
    sales_df.head(50),
    hide_index=True,
    on_select="rerun",
    selection_mode="multi-row",
    height=300,
    column_config={
        "Revenue": st.column_config.NumberColumn("Revenue", format="$ %d"),
    },
)

if selection and selection["selection"]["rows"]:
    selected_rows = sales_df.head(50).iloc[selection["selection"]["rows"]]
    st.success(f"Selected **{len(selected_rows)}** rows")

    c1, c2, c3 = st.columns(3)
    c1.metric("Total Revenue", f"${selected_rows['Revenue'].sum():,.0f}")
    c2.metric("Avg Rating", f"{selected_rows['Rating'].mean():.1f}")
    c3.metric("Total Units", f"{selected_rows['Units'].sum():,}")

    st.dataframe(selected_rows, hide_index=True)

---

## 🔧 Build It: Pandas Styler Integration

Conditional formatting makes patterns visible at a glance.

In [ ]:
st.header("Conditional Formatting with Styler")

# Color-code revenue: green for high, red for low
def color_revenue(val):
    if val >= 10000:
        return "background-color: #c6efce; color: #006100"
    elif val >= 5000:
        return "background-color: #ffeb9c; color: #9c6500"
    else:
        return "background-color: #ffc7ce; color: #9c0006"

# Color-code rating
def color_rating(val):
    if val >= 4.0:
        return "background-color: #c6efce"
    elif val >= 3.0:
        return "background-color: #ffeb9c"
    else:
        return "background-color: #ffc7ce"

sample = sales_df.head(20)[["Product", "Region", "Revenue", "Units", "Rating"]].copy()
styled = sample.style.applymap(color_revenue, subset=["Revenue"])
styled = styled.applymap(color_rating, subset=["Rating"])
styled = styled.format({"Revenue": "${:,.0f}", "Rating": "{:.1f}"})

st.dataframe(styled)

st.caption("Green = strong performance, Yellow = moderate, Red = weak.")

---

## 🧪 Experiment: Sidebar Filters — Transform Then Display

This is the core pattern: **filter → transform → display**.

In [ ]:
st.header("🧪 Interactive Filtering")

# Sidebar controls
with st.sidebar:
    st.header("🔍 Filters")

    # Product filter
    selected_products = st.multiselect(
        "Products",
        options=sales_df["Product"].unique().tolist(),
        default=sales_df["Product"].unique().tolist(),
    )

    # Region filter
    selected_regions = st.multiselect(
        "Regions",
        options=sales_df["Region"].unique().tolist(),
        default=sales_df["Region"].unique().tolist(),
    )

    # Revenue range
    min_revenue, max_revenue = st.slider(
        "Revenue range",
        min_value=int(sales_df["Revenue"].min()),
        max_value=int(sales_df["Revenue"].max()),
        value=(int(sales_df["Revenue"].min()), int(sales_df["Revenue"].max())),
    )

    # Min rating
    min_rating = st.slider("Minimum rating", 1.0, 5.0, 1.0, 0.1)

# Apply filters
filtered = sales_df[
    (sales_df["Product"].isin(selected_products)) &
    (sales_df["Region"].isin(selected_regions)) &
    (sales_df["Revenue"] >= min_revenue) &
    (sales_df["Revenue"] <= max_revenue) &
    (sales_df["Rating"] >= min_rating)
]

st.write(f"**Showing {len(filtered)} of {len(sales_df)} records**")

st.dataframe(
    filtered,
    hide_index=True,
    column_config={
        "Revenue": st.column_config.NumberColumn("Revenue", format="$ %d"),
        "Unit Price": st.column_config.NumberColumn("Unit Price", format="$ %.2f"),
        "Rating": st.column_config.NumberColumn("Rating", format="%.1f"),
    },
    height=400,
)

---

## 🧪 Experiment: Aggregation & Pivot Tables

Transform raw data into summaries that tell a story.

In [ ]:
st.header("🧪 Aggregation Patterns")

tab_group, tab_pivot, tab_computed = st.tabs(["GroupBy", "Pivot Table", "Computed Columns"])

with tab_group:
    st.subheader("Revenue by Region and Product")
    region_product = sales_df.groupby(["Region", "Product"]).agg({
        "Revenue": "sum",
        "Units": "sum",
        "Rating": "mean",
    }).reset_index()
    region_product["Revenue"] = region_product["Revenue"].round(0)
    region_product["Rating"] = region_product["Rating"].round(2)

    st.dataframe(
        region_product.sort_values("Revenue", ascending=False),
        hide_index=True,
        column_config={
            "Revenue": st.column_config.NumberColumn("Revenue", format="$ %d"),
            "Rating": st.column_config.NumberColumn("Avg Rating", format="%.2f"),
        },
    )

with tab_pivot:
    st.subheader("Revenue Pivot Table")
    pivot = sales_df.pivot_table(
        values="Revenue",
        index="Region",
        columns="Product",
        aggfunc="sum",
        fill_value=0,
    )
    st.dataframe(
        pivot,
        column_config={col: st.column_config.NumberColumn(format="$ %d") for col in pivot.columns},
    )

with tab_computed:
    st.subheader("Computed Columns")
    computed = sales_df[["Product", "Region", "Revenue", "Units", "Rating"]].copy()
    computed["Profit Margin"] = ((computed["Revenue"] * 0.3) / computed["Revenue"] * 100).round(1)
    computed["Revenue per Unit"] = (computed["Revenue"] / computed["Units"]).round(2)
    computed["Performance"] = computed["Rating"].apply(
        lambda x: "🟢 Strong" if x >= 4.0 else ("🟡 Moderate" if x >= 3.0 else "🔴 Weak")
    )

    st.dataframe(
        computed.head(20),
        hide_index=True,
        column_config={
            "Revenue": st.column_config.NumberColumn("Revenue", format="$ %d"),
            "Revenue per Unit": st.column_config.NumberColumn("Rev/Unit", format="$ %.2f"),
            "Profit Margin": st.column_config.NumberColumn("Margin", format="%.1f%%"),
        },
    )

---

## ⚠️ Common Mistakes

### Mistake 1: Using st.write for Tables

```python
# ❌ No formatting, no interactivity
st.write(df)

# ✅ Interactive, formatted, sortable
st.dataframe(df, column_config={...})
```

### Mistake 2: Showing Raw Large DataFrames

```python
# ❌ 100,000 rows — slow and overwhelming
st.dataframe(huge_df)

# ✅ Filter first, then display
filtered = huge_df[huge_df["Year"] == 2026]
st.dataframe(filtered)
```

### Mistake 3: Ignoring Column Config

```python
# ❌ 1234567.891234
# ✅ $1,234,567.89
st.dataframe(df, column_config={
    "Revenue": st.column_config.NumberColumn("Revenue", format="$%,.2f")
})
```

---

## 🔍 Debugging Tips

| Symptom | Likely Cause | Fix |
|---|---|---|
| Numbers look ugly | Missing column_config | Add `NumberColumn(format=...)` |
| Index column visible | hide_index not set | Add `hide_index=True` |
| Table too wide | Not using width control | Add `width="stretch"` or `use_container_width=True` |
| Styler not showing colors | Using st.write instead of st.dataframe | Pass Styler to `st.dataframe()` |
| Empty table | Filter removed all rows | Check filter logic, add `len(filtered) > 0` guard |
| Slow rendering | Too many rows | Filter, aggregate, or use `height` parameter |

---

## ✅ Best Practices

1. **Always use `hide_index=True`** for clean presentation.
2. **Always use `column_config`** to format numbers, dates, and percentages.
3. **Filter before display** — sidebar controls → filtered DataFrame → `st.dataframe()`.
4. **Aggregate for summaries** — don't show raw data when a summary tells the story.
5. **Use `height` parameter** to control how many rows are visible.
6. **Use `column_order`** to show the most important columns first.
7. **Use `on_select`** to make DataFrames interactive — let users select and explore.
8. **Use Styler** for conditional formatting when color coding matters.
9. **Use `width="stretch"`** (or the deprecated `use_container_width=True`) to fill available space.
10. **Add `help` text** to column_config for non-obvious columns.

---

## ✏️ Exercises

### Exercise 1: Formatted Employee Table
Create an employee DataFrame with columns: Name, Department, Salary, Start Date, Performance Rating. Display it with:
- Salary formatted as currency
- Start Date formatted as "MMM D, YYYY"
- Rating formatted as "X.X ⭐"
- Hidden index
- Sorted by Salary descending

### Exercise 2: Filtered Customer View
Build a customer dataset (100 rows) with: Name, City, Purchase Amount, Date, Category. Add sidebar filters for City (multiselect) and Purchase Amount (slider). Display filtered results with formatted columns.

### Exercise 3: Conditional Formatting Report
Create a sales report with:
- Revenue column colored green (≥$10K), yellow (≥$5K), red (<$5K)
- A computed "Performance" column (Strong/Moderate/Weak based on rating)
- GroupBy summary by Region with formatted totals

## 🚀 Challenge Problem

Build a **Sales Performance Dashboard** that:
1. Generates a 500-row sales dataset with Product, Region, Category, Revenue, Units, Rating, Date
2. Has sidebar filters: Product (multiselect), Region (multiselect), Revenue range (slider), Min rating (slider)
3. Shows a formatted `st.dataframe` with column_config for all numeric/date columns
4. Includes a Styler with conditional color coding on Revenue and Rating
5. Shows a Pivot Table of Revenue by Region × Product
6. Has a GroupBy summary table sorted by total Revenue
7. Enables row selection to show details of selected transactions

Use `np.random.seed(42)` for reproducibility.

---

## 📌 Key Takeaways

1. **`st.dataframe()`** is the primary tool for interactive tables — always use it over `st.write()`.
2. **`column_config`** transforms raw numbers into meaningful information (currency, dates, percentages).
3. **`hide_index=True`** makes tables clean and professional.
4. **Row selection** (`on_select="rerun"`) makes tables interactive — users can explore details.
5. **Pandas Styler** adds conditional formatting — color-code good/bad values.
6. **Filter before display** — the sidebar is your control panel, the main area is your results.
7. **Aggregate and pivot** — summaries tell stories better than raw data.

---

## 📚 Further Reading

- [st.dataframe API Reference](https://docs.streamlit.io/develop/api-reference/data/st.dataframe)
- [st.column_config API Reference](https://docs.streamlit.io/develop/api-reference/data/st.column_config)
- [Dataframes Guide](https://docs.streamlit.io/develop/concepts/design/dataframes)
- [Pandas Styler Documentation](https://pandas.pydata.org/docs/user_guide/style.html)

---

## 🔗 Related Materials

- 📖 Reading: [07 — Data Display: DataFrames, Tables & Pandas Integration](../readings/07_data_display_dataframes.md)
- 📖 Reading: [08 — Visualization with Streamlit, Matplotlib & Plotly](../readings/08_visualization_matplotlib_plotly.md)
- 📓 Notebook: [08 — Interactive Visualization & Chart Selection](08_interactive_visualization.ipynb)
- ✏️ Exercise: [07 — Data Display Challenges](../exercises/07_data_display_challenges.py)
- 🖥️ Demo App: [07 — Data Display Demo](../apps/07_data_display_demo.py)
- 📝 Quiz: [04 — DataFrames & Visualization](../quizzes/04_dataframes_visualization.md)